# ETL Fase 3: Procesamiento de Tablas Maestras
Este notebook tiene como objetivo realizar el proceso de Extracción, Transformación y Carga (ETL) de los datos estáticos o catálogos principales del proyecto (Torneos y Clubes). 

## 1. Importación de Librerías y Configuración del Entorno
En esta primera etapa, importamos las herramientas necesarias para la manipulación de datos, la gestión de rutas del sistema y la conexión segura a nuestra base de datos en SQL Server.

In [ ]:
# Herramienta principal para la manipulación y análisis de datos en tablas (DataFrames)
import pandas as pd

# Herramientas del sistema operativo para manipular rutas de archivos y carpetas
import os
import sys

# Librerías necesarias para crear la conexión a la base de datos SQL Server
import urllib.parse
from sqlalchemy import create_engine

# Herramientas para leer credenciales secretas desde un archivo .env (evita exponer contraseñas en el código)
from dotenv import load_dotenv, find_dotenv

# Configuramos el sistema para que reconozca la carpeta principal del proyecto.
# Esto es útil si más adelante necesitamos importar otros scripts de Python creados por nosotros.
sys.path.append(os.path.abspath('..'))

print("Librerías importadas correctamente. El entorno está listo para trabajar.")

Librerías importadas correctamente.


## 2. Configuración y Conexión Segura a la Base de Datos
En este paso, establecemos la conexión con nuestro servidor de SQL Server. Siguiendo los estándares de seguridad de la industria, evitamos escribir ("hardcodear") los nombres del servidor y la base de datos directamente en el código. En su lugar, extraemos estos valores de un archivo de entorno oculto (`.env`). Finalmente, construimos un "motor" (engine) que actuará como el puente de comunicación entre nuestro entorno de Python y la base de datos.

In [ ]:
# 1. Buscamos y cargamos automáticamente el archivo oculto .env que contiene nuestras configuraciones locales
load_dotenv(find_dotenv())

# 2. Extraemos las variables de entorno almacenadas (Servidor, Base de Datos y Controlador)
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')
driver = os.getenv('DB_DRIVER')

# 3. Control de seguridad: Verificamos que el sistema haya logrado leer el archivo correctamente.
# Si los valores están vacíos, el programa se detiene y nos alerta del error.
if not server or not database:
    raise ValueError("Error: No se pudieron cargar las configuraciones de la base de datos desde el archivo .env")

# 4. Formateamos la cadena de conexión de manera segura. 
# La instrucción 'Trusted_Connection=yes' indica que usaremos la autenticación nativa de Windows (sin contraseñas explícitas).
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

# 5. Creamos el 'engine' (motor), que es la herramienta de SQLAlchemy encargada de ejecutar las operaciones hacia SQL Server
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

print("¡Motor de base de datos configurado dinámicamente y de forma segura!")

¡Motor de base de datos configurado dinámicamente desde el archivo .env!


## 3. Extracción de Datos Crudos (Extract)
En esta etapa, le indicamos al programa dónde se encuentran los archivos originales descargados de Kaggle. Utilizamos la librería `pandas` para leer estos archivos `.csv` y transformarlos en "DataFrames" (que son esencialmente tablas virtuales bidimensionales en memoria, muy similares a las hojas de cálculo de Excel). El sufijo `_raw` (crudo) en el nombre de las variables nos ayuda a recordar que esta información aún no ha sido limpiada ni procesada.

In [ ]:
# 1. Definimos la ruta relativa hacia la carpeta donde guardamos los archivos originales.
# Usar rutas relativas ('../') permite que el código funcione en otras computadoras sin importar dónde guarden el proyecto.
raw_data_path = '../data/raw/'

# 2. Cargamos los archivos CSV en memoria como DataFrames de pandas.
# Usamos 'os.path.join' para unir la ruta de la carpeta con el nombre exacto del archivo, 
# lo cual evita errores si ejecutamos el código en Mac, Linux o Windows (que usan barras de ruta diferentes).

df_tournaments_raw = pd.read_csv(os.path.join(raw_data_path, '01_EWC2025_Event_Tournament_Summary.csv'))
df_partners_raw = pd.read_csv(os.path.join(raw_data_path, '04_EWC2025_Club_Partner_Program.csv'))
df_standings_raw = pd.read_csv(os.path.join(raw_data_path, '03_EWC2025_Club_Championship_Standings.csv'))
df_players_raw = pd.read_csv(os.path.join(raw_data_path, '05_EWC2025_Player_Roster.csv'))

# 3. Mensaje de validación para confirmar que la extracción fue exitosa
print("Archivos CSV extraídos a memoria con éxito y listos para ser transformados.")

Archivos CSV extraídos a memoria con éxito.


## 4. Transformación de la Tabla Maestra: Torneos (Transform)
En esta fase iniciamos el proceso de transformación aplicada al catálogo de torneos. El archivo original de Kaggle contiene múltiples filas para un mismo videojuego si este se dividió en varias etapas o sub-eventos competitivos. Dado que en nuestro diseño relacional el título del videojuego (`Game_Title`) actúa como la Llave Primaria, la base de datos rechazaría cualquier duplicado de nombre. Para resolver esta inconsistencia de forma masiva, renombramos las columnas con una nomenclatura estándar y ejecutamos una operación de agrupación. Mediante reglas matemáticas específicas, consolidamos los registros repetidos sumando las bolsas de premios y calculando los rangos de fechas correctos, garantizando así un catálogo limpio con registros únicos listo para la inserción.

In [ ]:
# 1. Renombrar columnas
# Modificamos el nombre de la columna original 'Game' por 'Game_Title' para que coincida exactamente 
# con la estructura de la tabla destino en SQL Server, manteniendo el resto de nombres de forma explícita.
df_tournaments = df_tournaments_raw.rename(columns={
    'Game': 'Game_Title', 
    'Event_Name': 'Event_Name', 
    'Start_Date': 'Start_Date', 
    'End_Date': 'End_Date', 
    'Prize_Pool_USD': 'Prize_Pool_USD', 
    'Num_Participants': 'Num_Participants', 
    'Game_Type': 'Game_Type', 
    'Platform': 'Platform'
})

# 2. Agrupamos por Game_Title (nuestra Llave Primaria)
# Como un videojuego puede aparecer varias veces, usamos '.groupby' para fusionar las filas duplicadas en una sola.
# Mediante '.agg' (agregación) definimos una regla lógica para indicarle a Python cómo consolidar los datos repetidos:
df_tournaments_grouped = df_tournaments.groupby('Game_Title', as_index=False).agg({
    'Event_Name': 'first',      # Conserva el texto del primer sub-evento que encuentre
    'Start_Date': 'min',        # Analiza las filas duplicadas y conserva la fecha de inicio más antigua
    'End_Date': 'max',          # Analiza las filas duplicadas y conserva la fecha de finalización más reciente
    'Prize_Pool_USD': 'sum',    # Suma el dinero de todos los premios repartidos en ese videojuego
    'Num_Participants': 'sum',  # Suma la cantidad total de los participantes de todas las etapas
    'Game_Type': 'first',       # Mantiene la categoría o género del juego (ej. Shooter, MOBA)
    'Platform': 'first'         # Mantiene la plataforma principal registrada
})

# 3. Filtrar estrictamente las columnas en el orden que van a la BD
# Creamos una lista con el orden idéntico al esquema de SQL Server y filtramos el DataFrame.
# Esta práctica previene errores críticos de inserción provocados por desorden de columnas en memoria.
columnas_tournaments = ['Game_Title', 'Event_Name', 'Start_Date', 'End_Date', 'Prize_Pool_USD', 'Num_Participants', 'Game_Type', 'Platform']
df_tournaments_final = df_tournaments_grouped[columnas_tournaments]

# 4. Verificación de Control de Calidad
# Imprimimos en consola la cantidad final de videojuegos únicos que pasaron la regla de validación
print(f"Total de Torneos/Juegos únicos listos para cargar: {len(df_tournaments_final)}")

# Desplegamos las primeras 5 filas en pantalla para una inspección visual de los datos transformados
df_tournaments_final.head(5)

Total de Torneos/Juegos únicos listos para cargar: 25


,Game_Title,Event_Name,Start_Date,End_Date,Prize_Pool_USD,Num_Participants,Game_Type,Platform
0,Apex Legends,ALGS 2025 Midseason Playoffs,2025-07-10,2025-07-13,2000000,40,Battle Royale,PC/Console
1,Call of Duty: Black Ops 6,Esports World Cup 2025,2025-07-24,2025-07-27,1800000,16,FPS,Console/PC
2,Call of Duty: Warzone,Esports World Cup 2025,2025-08-06,2025-08-09,1000000,21,Battle Royale,Console/PC
3,Chess,Esports World Cup 2025,2025-07-29,2025-08-01,1500000,16,Strategy,Online
4,Counter-Strike 2,Esports World Cup 2025,2025-08-20,2025-08-24,1250000,16,FPS,PC


## 5. Consolidación de la Tabla Maestra: Clubes (Transform)
En esta fase abordamos la transformación del catálogo de Clubes. A diferencia de los torneos, la información de las organizaciones se encuentra dispersa a lo largo de varios archivos originales, ya que algunos clubes participan en el programa de socios (partners), otros figuran en la tabla de posiciones y otros solo aparecen en el listado de jugadores. Para garantizar la integridad referencial del modelo de base de datos, es obligatorio construir una lista maestra exhaustiva. El proceso consiste en extraer los nombres de las organizaciones de todas las fuentes disponibles, eliminar los valores nulos, concatenar los resultados y aislar únicamente los valores únicos. Una vez establecida esta "fuente única de verdad", cruzamos la información mediante un 'Left Join' con el archivo del programa de socios para enriquecer el catálogo con metadatos corporativos, como el año de fundación, la región y el CEO, asumiendo valores nulos para aquellas organizaciones que no cuentan con este perfil extendido.

In [ ]:
# 1. Recolectar nombres únicos de clubes de múltiples fuentes
# Extraemos las columnas que contienen los nombres de los equipos en cada archivo.
# Utilizamos el método '.dropna()' de inmediato para descartar cualquier fila vacía que pueda generar ruido.
clubes_partners = df_partners_raw['Organization'].dropna()
clubes_standings = df_standings_raw['Organization'].dropna()
clubes_players = df_players_raw['Team'].dropna()

# 2. Concatenar y obtener valores únicos reales
# Unimos matemáticamente las tres series de datos en una sola columna masiva usando 'pd.concat'.
# Posteriormente, aplicamos '.unique()' para eliminar todos los nombres repetidos, obteniendo el catálogo definitivo.
clubes_unicos = pd.concat([clubes_partners, clubes_standings, clubes_players]).unique()

# 3. Crear el DataFrame maestro
# Transformamos nuestro arreglo de nombres únicos en un DataFrame formal de pandas, 
# asignándole el nombre de columna oficial que actuará como Llave Primaria en SQL Server.
df_clubs_master = pd.DataFrame({'Organization_Name': clubes_unicos})

# 4. Preparar la tabla de Partners para el cruce
# Renombramos las columnas del archivo de socios corporativos para estandarizar la nomenclatura 
# antes de intentar unirlo con nuestro listado maestro.
df_partners_clean = df_partners_raw.rename(columns={
    'Organization': 'Organization_Name', 
    'Region': 'Region', 
    'Founded': 'Founded_Year', 
    'CEO': 'CEO', 
    'Social_Media_Followers_M': 'Social_Media_Followers_M'
})

# 5. Cruce relacional (Left Join)
# Utilizamos 'pd.merge' para fusionar el catálogo maestro con los metadatos corporativos.
# El parámetro 'how="left"' es crítico aquí: asegura que conservaremos TODOS los clubes de nuestra lista maestra, 
# agregando la información de la región o el CEO solo a aquellos clubes que sí existan en el archivo de Partners.
df_clubs = pd.merge(
    df_clubs_master, 
    df_partners_clean[['Organization_Name', 'Region', 'Founded_Year', 'CEO', 'Social_Media_Followers_M']], 
    on='Organization_Name', 
    how='left'
)

# Imprimimos en consola el conteo final para verificar la volumetría de la dimensión
print(f"Total de Clubes únicos consolidados: {len(df_clubs)}")

# Desplegamos una muestra visual de la estructura final
df_clubs.head(5)

Total de Clubes únicos consolidados: 71


,Organization_Name,Region,Founded_Year,CEO,Social_Media_Followers_M
0,Fnatic,Europe,2004.0,Sam Mathews,2.5
1,G2 Esports,Europe,2013.0,Alban Dechelotte,3.2
2,Gentle Mates,Europe,2023.0,Squeezie,1.8
3,HEROIC,Europe,2016.0,Joachim Haraldsen,0.9
4,Karmine Corp,Europe,2020.0,Kameto,2.1


## 6. Ingesta de Datos en SQL Server (Load)
Llegamos a la fase final del proceso ETL: la carga física de la información en nuestro motor de base de datos relacional. En lugar de ejecutar comandos repetitivos, diseñamos una función de carga modular y segura. Esta función utiliza el motor de conexión (`engine`) configurado al inicio para inyectar los DataFrames directamente en las tablas correspondientes. Es de suma importancia destacar el uso del parámetro `append`; esto le indica a pandas que debe insertar los nuevos registros respetando estrictamente el esquema, los tipos de datos y las llaves primarias que ya definimos en SQL Server, evitando que la librería intente sobrescribir o alterar la arquitectura estructural. Además, el bloque de código cuenta con un mecanismo de captura de errores (Try-Except) para notificar cualquier fallo de integridad sin interrumpir la ejecución total del programa.

In [ ]:
# 1. Definición de la función de carga estandarizada
# Recibe tres parámetros: el DataFrame a cargar, el nombre exacto de la tabla en SQL Server y el motor de conexión.
def cargar_tabla_maestra(df, table_name, engine):
    print(f"Iniciando carga de tabla: {table_name}...")
    
    # Iniciamos un bloque Try-Except. Esto es una buena práctica de ingeniería defensiva:
    # Si la base de datos rechaza la inserción (ej. por un duplicado), el código captura el error,
    # lo imprime en pantalla para su análisis, pero permite que el script continúe ejecutándose.
    try:
        # El método '.to_sql' traduce el DataFrame de pandas a comandos INSERT de SQL.
        # if_exists='append': Añade los datos a la tabla existente SIN modificar su estructura original.
        # index=False: Evita que el índice numérico automático de pandas se envíe como una columna extra.
        df.to_sql(name=table_name, con=engine, if_exists='append', index=False)
        print(f"  -> Éxito: {len(df)} registros insertados de forma segura en dbo.{table_name}.\n")
        
    except Exception as e:
        # Si algo falla, el programa salta a este bloque y nos muestra el mensaje oficial de error de SQL Server
        print(f"  -> ERROR CRÍTICO al cargar la tabla {table_name}:")
        print(e)
        print("\n")

# 2. Ejecución de la ingesta
# Llamamos a nuestra función pasando los DataFrames finales que limpiamos en los pasos anteriores.
# Al ser tablas maestras (dimensiones independientes), el orden de carga entre ellas no afecta la integridad referencial.
cargar_tabla_maestra(df_tournaments_final, 'Tournaments', engine)
cargar_tabla_maestra(df_clubs, 'Clubs', engine)

# Mensaje de cierre para certificar el éxito del script completo
print("ETL de Tablas Maestras completado satisfactoriamente. Los catálogos están listos en la base de datos.")

Iniciando carga de tabla: Tournaments...
  -> Éxito: 25 registros insertados en dbo.Tournaments.

Iniciando carga de tabla: Clubs...
  -> Éxito: 71 registros insertados en dbo.Clubs.

ETL de Tablas Maestras completado satisfactoriamente.
